# Moirai Classifier for Kalshi Jump Prediction

This notebook uses a pretrained Salesforce Moirai / Moirai-MoE time-series backbone as a sequence encoder, replaces the forecasting distribution head with a multiclass classification head, and predicts four jump horizons simultaneously. It stays at trade level: no bars, no resampling, no forward filling, and no feature recomputation.

In [1]:
# Cell 1 - Setup
import os
import math
import random
import gc
from google.colab import drive

DRIVE_ROOT = '/content/drive'
DRIVE_MYDRIVE = os.path.join(DRIVE_ROOT, 'MyDrive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
if os.path.isdir(DRIVE_MYDRIVE):
    print(f'Drive already available at {DRIVE_ROOT}')
else:
    mount_errors = []
    for force in [False, True]:
        try:
            drive.mount(DRIVE_ROOT, force_remount=force, timeout_ms=300000)
        except Exception as exc:
            mount_errors.append(f'force_remount={force}: {type(exc).__name__}: {exc}')
        if os.path.isdir(DRIVE_MYDRIVE):
            print(f'Drive mounted at {DRIVE_ROOT}')
            break
    else:
        print('Drive mount attempts failed:')
        for err in mount_errors:
            print('  ', err)
        print('Current /content/drive contents:', os.listdir(DRIVE_ROOT) if os.path.isdir(DRIVE_ROOT) else 'missing')
        raise RuntimeError('Google Drive is not mounted in this Colab VM. Restart the runtime and complete the Drive auth prompt.')

# No automatic pip installs or forced runtime restarts here. If a dependency is
# missing, install it manually in a separate cell, restart once, then rerun Cell 1.
import numpy as np
import pandas as pd
import pyarrow as pa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

print('Dependency versions:')
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('pyarrow:', pa.__version__)
print('torch:', torch.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

PARQUET_PATH = '/content/drive/MyDrive/CS1090B/project/data/all_trades_features.parquet'
CHECKPOINT_DIR = '/content/drive/MyDrive/CS1090B/project/checkpoints'
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'moirai_classifier_best.pt')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


Mounted at /content/drive
Drive mounted at /content/drive
Dependency versions:
numpy: 2.0.2
pandas: 2.2.2
pyarrow: 18.1.0
torch: 2.10.0+cu128
Using device: cuda
NVIDIA A100-SXM4-80GB


In [2]:
# Cell 1b - Install Moirai/Uni2TS dependencies if missing
# Run this once in a fresh Colab runtime before Cell 5.
# Uses try/import so it reinstalls if packages are broken or missing (not just absent metadata).
import sys
import subprocess

try:
    from uni2ts.model.moirai import MoiraiModule
    from uni2ts.model.moirai_moe import MoiraiMoEModule
    from uni2ts.common.torch_util import packed_attention_mask, packed_causal_attention_mask
    print('uni2ts/Moirai imports OK')
except Exception as exc:
    print('Installing uni2ts/Moirai dependencies...')
    print('Initial import error:', repr(exc))
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-cache-dir', '--no-deps',
        'uni2ts==2.0.0'
    ])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-cache-dir',
        'einops==0.7.0',
        'jaxtyping~=0.2.24',
        'gluonts==0.14.3',
        'hydra-core==1.3',
        'python-dotenv==1.0.0',
        'orjson',
        'tensorboard',
        'multiprocess',
        'safetensors',
        'datasets==2.17.1',
        'huggingface_hub',
        'lightning',
        'toolz',
    ])
    # Purge the pip build cache to reclaim disk space after installation.
    subprocess.call([sys.executable, '-m', 'pip', 'cache', 'purge'])
    from uni2ts.model.moirai import MoiraiModule
    from uni2ts.model.moirai_moe import MoiraiMoEModule
    from uni2ts.common.torch_util import packed_attention_mask, packed_causal_attention_mask
    print('uni2ts/Moirai imports OK after install')

Installing uni2ts/Moirai dependencies...
Initial import error: ModuleNotFoundError("No module named 'uni2ts'")
uni2ts/Moirai imports OK after install


In [3]:
# Cell 2 - Load preprocessed trade-level data
df = pd.read_parquet(PARQUET_PATH)

TARGET_COLUMNS = ['jump3_5m', 'jump3_15m', 'jump3_30m', 'jump3_60m']
HORIZON_NAMES = ['5m', '15m', '30m', '60m']
CLASS_NAMES = ['down', 'none', 'up']

# Must match FEATURE_COLS in preprocessing_clean_m4.ipynb.
FEATURE_COLUMNS = [
    'yes_price', 'log_volume', 'log_time_delta',
    'ret_last_5', 'ret_last_10', 'ret_last_20',
    'position_in_range', 'dist_from_50',
    'relative_volume', 'tick', 'run_length',
    'volume_since_price_change',
    'trades_last_60s', 'trades_last_300s',
    'hour_sin', 'hour_cos',
    'backward_ret_5m', 'backward_ret_15m', 'backward_ret_30m', 'backward_ret_60m',
    'backward_abs_ret_5m', 'backward_abs_ret_15m', 'backward_abs_ret_30m', 'backward_abs_ret_60m',
    'recent_up_jumps_5m', 'recent_up_jumps_15m', 'recent_up_jumps_30m', 'recent_up_jumps_60m',
    'recent_down_jumps_5m', 'recent_down_jumps_15m', 'recent_down_jumps_30m', 'recent_down_jumps_60m',
    *[f'ticker_emb_{i}' for i in range(8)],
]

missing = [c for c in ['ticker', 'created_time', *FEATURE_COLUMNS, *TARGET_COLUMNS] if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
assert len(FEATURE_COLUMNS) == 40

if pd.api.types.is_numeric_dtype(df['created_time']):
    date_series = pd.to_datetime(df['created_time'], unit='us', utc=True, errors='coerce')
else:
    date_series = pd.to_datetime(df['created_time'], utc=True, errors='coerce')

print('Shape:', df.shape)
print('Date range:', date_series.min(), '->', date_series.max())
print('Unique tickers:', df['ticker'].nunique())
print('Feature count:', len(FEATURE_COLUMNS))

print('\nTop feature NaN counts:')
print(df[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False).head(15).to_string())

print('\nClass balance by target:')
for col in TARGET_COLUMNS:
    out = pd.DataFrame({
        'count': df[col].value_counts(dropna=False).sort_index(),
        'pct': (df[col].value_counts(normalize=True, dropna=False).sort_index() * 100).round(3),
    })
    print(f'\n{col}')
    print(out)

Shape: (63801798, 63)
Date range: 2025-01-01 00:00:09.673219+00:00 -> 2025-11-25 22:00:15.194245+00:00
Unique tickers: 517520
Feature count: 40

Top feature NaN counts:
position_in_range            6346469
ret_last_20                  4109982
ret_last_10                  2545664
ret_last_5                   1552454
relative_volume               523692
log_time_delta                     0
yes_price                          0
log_volume                         0
dist_from_50                       0
tick                               0
run_length                         0
volume_since_price_change          0
trades_last_60s                    0
trades_last_300s                   0
hour_sin                           0

Class balance by target:

jump3_5m
             count     pct
jump3_5m                  
-1.0       5422472   8.499
 0.0      52928448  82.958
 1.0       5450878   8.543

jump3_15m
              count     pct
jump3_15m                  
-1.0        6039084   9.465
 0.0      

In [4]:
# Cell 3 - Temporal train/val/test split
# If preprocessing saved only train/test, validation is carved from the end of train.
ORIGINAL_N = len(df)
VAL_FRACTION_OF_PREPROCESS_TRAIN = 0.10

if 'split' in df.columns:
    split_col = df['split'].astype(str).str.lower()
    if split_col.eq('val').any():
        train_mask = split_col.eq('train')
        val_mask = split_col.eq('val')
        test_mask = split_col.eq('test')
        print('Using saved train/val/test split.')
    else:
        pre_train_mask = split_col.eq('train')
        test_mask = split_col.eq('test')
        val_cutoff = date_series.loc[pre_train_mask].quantile(1.0 - VAL_FRACTION_OF_PREPROCESS_TRAIN)
        train_mask = pre_train_mask & (date_series < val_cutoff)
        val_mask = pre_train_mask & (date_series >= val_cutoff)
        print('Using saved preprocessing train/test split; carving val from end of train.')
else:
    train_cutoff = date_series.quantile(0.70)
    val_cutoff = date_series.quantile(0.80)
    train_mask = date_series < train_cutoff
    val_mask = (date_series >= train_cutoff) & (date_series < val_cutoff)
    test_mask = date_series >= val_cutoff
    print('No split column found; using fallback 70/10/20 temporal split.')

if not train_mask.any() or not val_mask.any() or not test_mask.any():
    raise ValueError('Empty train/val/test split. Check split and created_time.')

MODEL_COLUMNS = ['ticker', 'created_time', *FEATURE_COLUMNS, *TARGET_COLUMNS]
train_df = df.loc[train_mask, MODEL_COLUMNS].copy()
val_df = df.loc[val_mask, MODEL_COLUMNS].copy()
test_df = df.loc[test_mask, MODEL_COLUMNS].copy()

print('Train end:', date_series.loc[train_mask].max())
print('Val:', date_series.loc[val_mask].min(), '->', date_series.loc[val_mask].max())
print('Test start:', date_series.loc[test_mask].min())
print('Trades:', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
print('Percent:', {
    'train': round(100 * len(train_df) / ORIGINAL_N, 3),
    'val': round(100 * len(val_df) / ORIGINAL_N, 3),
    'test': round(100 * len(test_df) / ORIGINAL_N, 3),
})

del df
gc.collect()

Using saved preprocessing train/test split; carving val from end of train.
Train end: 2025-10-29 02:41:31.136746+00:00
Val: 2025-10-29 02:41:31.323698+00:00 -> 2025-11-06 03:58:08.005510+00:00
Test start: 2025-11-06 03:58:08.028990+00:00
Trades: {'train': 45937294, 'val': 5104144, 'test': 12760360}
Percent: {'train': 72.0, 'val': 8.0, 'test': 20.0}


0

In [5]:
# Cell 4 - Vectorized sequence dataset
# IMPORTANT: every window is same-ticker only. Moirai is used as an encoder here.
# It does not forecast the next N trades/bars; labels define the future real-time horizons.
# This implementation is vectorized over the whole split and avoids DataLoader workers,
# which were duplicating huge Python objects and stalling before GPU training.

class KalshiMoiraiDataset:
    def __init__(self, frame, feature_cols, target_cols, seq_len, stride, feature_mean, feature_std, max_sequences=None, seed=42, name='dataset'):
        self.feature_cols = list(feature_cols)
        self.target_cols = list(target_cols)
        self.seq_len = int(seq_len)
        self.stride = int(stride)
        self.name = name
        self.window_offsets = np.arange(self.seq_len, dtype=np.int64)
        rng = np.random.default_rng(seed)

        print(f'[{self.name}] sorting rows by ticker/time...')
        frame = frame.sort_values(['ticker', 'created_time'], kind='mergesort').reset_index(drop=True)
        n_rows = len(frame)
        print(f'[{self.name}] rows: {n_rows:,}')

        print(f'[{self.name}] building normalized feature matrix...')
        feature_mean = np.asarray(feature_mean, dtype=np.float32)
        feature_std = np.asarray(feature_std, dtype=np.float32)
        x_raw = frame[self.feature_cols].to_numpy(dtype=np.float32, copy=True)
        self.X = ((x_raw - feature_mean) / feature_std).astype(np.float32, copy=False)
        del x_raw

        print(f'[{self.name}] building label matrix...')
        y_raw = frame[self.target_cols].to_numpy(copy=True)
        y_all = np.full(y_raw.shape, -100, dtype=np.int64)
        y_all[y_raw == -1] = 0
        y_all[y_raw == 0] = 1
        y_all[y_raw == 1] = 2

        valid_feature_rows = np.isfinite(self.X).all(axis=1)
        valid_label_rows = (~pd.isna(y_raw).any(axis=1)) & ((y_all >= 0).all(axis=1))
        valid_prefix = np.concatenate([[0], np.cumsum(valid_feature_rows, dtype=np.int32)])
        del y_raw

        print(f'[{self.name}] vectorizing same-ticker window starts...')
        ticker_codes = pd.factorize(frame['ticker'], sort=False)[0]
        change_points = np.where(np.diff(ticker_codes) != 0)[0] + 1
        group_starts = np.concatenate([[0], change_points])
        group_ends = np.concatenate([change_points, [n_rows]])

        starts_list = []
        label_list = []
        for s, e in zip(group_starts, group_ends):
            if e - s < self.seq_len:
                continue
            end_pos = np.arange(s + self.seq_len - 1, e, self.stride, dtype=np.int64)
            start_pos = end_pos - self.seq_len + 1
            valid_windows = (valid_prefix[end_pos + 1] - valid_prefix[start_pos]) == self.seq_len
            keep = valid_label_rows[end_pos] & valid_windows
            if keep.any():
                starts_list.append(start_pos[keep].astype(np.int64))
                label_list.append(y_all[end_pos[keep]].astype(np.int64))

        if starts_list:
            self.starts = np.concatenate(starts_list)
            self.labels = np.concatenate(label_list, axis=0)
        else:
            self.starts = np.empty(0, dtype=np.int64)
            self.labels = np.empty((0, len(self.target_cols)), dtype=np.int64)

        if max_sequences is not None and len(self.starts) > max_sequences:
            print(f'[{self.name}] subsampling windows: {len(self.starts):,} -> {int(max_sequences):,}')
            keep = rng.choice(len(self.starts), size=int(max_sequences), replace=False)
            self.starts = self.starts[keep]
            self.labels = self.labels[keep]

        print(f'[{self.name}] valid sequences: {len(self.starts):,}')
        gc.collect()

    def __len__(self):
        return len(self.starts)

    def label_counts(self):
        counts = np.zeros((len(self.target_cols), 3), dtype=np.int64)
        for h in range(len(self.target_cols)):
            counts[h] = np.bincount(self.labels[:, h], minlength=3)[:3]
        return counts

    def get_batch(self, batch_indices, device):
        batch_indices = np.asarray(batch_indices, dtype=np.int64)
        rows = self.starts[batch_indices, None] + self.window_offsets[None, :]
        x_np = self.X[rows]
        y_np = self.labels[batch_indices]
        x = torch.from_numpy(x_np).to(device, non_blocking=False)
        y = torch.from_numpy(y_np).to(device, non_blocking=False)
        return x, y


def iter_batches(dataset, batch_size, device, shuffle=False, seed=42):
    n = len(dataset)
    order = np.arange(n, dtype=np.int64)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(order)
    for start in range(0, n, batch_size):
        batch_idx = order[start:start + batch_size]
        yield start // batch_size + 1, dataset.get_batch(batch_idx, device)


In [6]:
# Cell 5 - Moirai backbone with classification head
from uni2ts.model.moirai import MoiraiModule
from uni2ts.model.moirai_moe import MoiraiMoEModule
from uni2ts.common.torch_util import packed_attention_mask, packed_causal_attention_mask

# For native forecasting, prediction_length would mean future steps in the series index.
# Since our data is irregular event-time trades, that would mean future trades, not minutes.
# This classifier avoids that issue: future 5/15/30/60m labels are supervised targets.
SEQ_LEN = 32
STRIDE = 16
PATCH_SIZE = 8       # Must divide SEQ_LEN and be supported by Moirai 1.x checkpoints.
BATCH_SIZE = 256
EPOCHS = 3 # low due to high amount of steps
LR_BACKBONE = 1e-5
LR_HEAD = 3e-4
WEIGHT_DECAY = 1e-2
WARMUP_STEPS = 500
MAX_SEQUENCES = None
RUN_TRAINING = False    # Default is checkpoint/export mode; set True only to retrain.
RUN_EVALUATION = False  # Set True to rebuild test_dataset and rerun test metrics/figures.
NUM_WORKERS = 0  # Not used by the custom iterator; kept for reference.
USE_AMP = True       # Mixed precision is needed for practical Moirai throughput on GPU.
USE_CLASS_WEIGHTS = True
FREEZE_BACKBONE = False
MOIRAI_KIND = 'moirai'  # 'moirai' or 'moirai-moe'
MOIRAI_SIZE = 'small'
MOIRAI_MODEL_ID = 'Salesforce/moirai-1.1-R-small'
# For Moirai-MoE small, use: MOIRAI_KIND='moirai-moe'; MOIRAI_MODEL_ID='Salesforce/moirai-moe-1.0-R-small'

if SEQ_LEN % PATCH_SIZE != 0:
    raise ValueError('SEQ_LEN must be divisible by PATCH_SIZE.')

class MoiraiBackboneClassifier(nn.Module):
    def __init__(self, module, input_dim, seq_len, patch_size, n_horizons=4, n_classes=3, kind='moirai'):
        super().__init__()
        self.module = module
        self.input_dim = int(input_dim)
        self.seq_len = int(seq_len)
        self.patch_size = int(patch_size)
        self.n_patches = self.seq_len // self.patch_size
        self.max_patch_size = int(max(module.patch_sizes))
        self.n_horizons = n_horizons
        self.n_classes = n_classes
        self.kind = kind
        if self.patch_size not in tuple(module.patch_sizes):
            raise ValueError(f'PATCH_SIZE={self.patch_size} not supported by checkpoint patch sizes {module.patch_sizes}')
        d_model = module.d_model
        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 3),
            nn.Linear(d_model * 3, d_model),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(d_model, n_horizons * n_classes),
        )

    def patchify(self, x):
        # x: (B, T, F). Moirai's MultiInSizeLinear expects the last dimension
        # to be max(module.patch_sizes), with patch_size telling it how much
        # of that vector is real. So we pad each length-PATCH_SIZE token.
        bsz, seq_len, n_feat = x.shape
        x = x.view(bsz, self.n_patches, self.patch_size, n_feat)
        x = x.permute(0, 1, 3, 2).contiguous()
        tokens = x.view(bsz, self.n_patches * n_feat, self.patch_size)
        if self.patch_size == self.max_patch_size:
            return tokens
        padded = tokens.new_zeros(bsz, self.n_patches * n_feat, self.max_patch_size)
        padded[..., :self.patch_size] = tokens
        return padded

    def encode(self, x):
        bsz = x.size(0)
        target = self.patchify(x)
        n_tokens = target.size(1)
        observed_mask = torch.ones_like(target, dtype=torch.bool)
        sample_id = torch.arange(bsz, device=x.device).unsqueeze(1).expand(bsz, n_tokens)
        time_id = torch.arange(self.n_patches, device=x.device).repeat_interleave(self.input_dim)
        time_id = time_id.unsqueeze(0).expand(bsz, n_tokens)
        variate_id = torch.arange(self.input_dim, device=x.device).repeat(self.n_patches)
        variate_id = variate_id.unsqueeze(0).expand(bsz, n_tokens)
        patch_size = torch.full((bsz, n_tokens), self.patch_size, dtype=torch.long, device=x.device)

        # We intentionally do not call the forecasting distribution head. This is representation learning.
        # Inputs are already standardized with train-only stats, so avoid a second per-window scaler
        # that could remove useful absolute price-level information.
        if self.kind == 'moirai-moe':
            in_reprs = self.module.in_proj(target, patch_size)
            in_reprs = F.silu(in_reprs)
            in_reprs = self.module.feat_proj(in_reprs, patch_size)
            res_reprs = self.module.res_proj(target, patch_size)
            reprs = in_reprs + res_reprs
            attn_mask = packed_causal_attention_mask(sample_id, time_id)
        else:
            reprs = self.module.in_proj(target, patch_size)
            attn_mask = packed_attention_mask(sample_id)

        reprs = self.module.encoder(reprs, attn_mask, time_id=time_id, var_id=variate_id)
        reprs = reprs.view(bsz, self.n_patches, self.input_dim, -1)
        last_patch = reprs[:, -1, :, :]
        yes_price_repr = last_patch[:, 0, :]
        mean_repr = last_patch.mean(dim=1)
        max_repr = last_patch.max(dim=1).values
        return torch.cat([yes_price_repr, mean_repr, max_repr], dim=-1)

    def forward(self, x):
        z = self.encode(x)
        logits = self.head(z)
        return logits.view(-1, self.n_horizons, self.n_classes)

if MOIRAI_KIND == 'moirai-moe':
    moirai_module = MoiraiMoEModule.from_pretrained(MOIRAI_MODEL_ID)
else:
    moirai_module = MoiraiModule.from_pretrained(MOIRAI_MODEL_ID)

model = MoiraiBackboneClassifier(
    moirai_module,
    input_dim=len(FEATURE_COLUMNS),
    seq_len=SEQ_LEN,
    patch_size=PATCH_SIZE,
    kind=MOIRAI_KIND,
).to(device)

if FREEZE_BACKBONE:
    for p in model.module.parameters():
        p.requires_grad_(False)

def count_parameters(m):
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    return trainable, total

trainable, total = count_parameters(model)
print(model)
print(f'Trainable parameters: {trainable:,} / {total:,}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/55.3M [00:00<?, ?B/s]

MoiraiBackboneClassifier(
  (module): MoiraiModule(
    (mask_encoding): Embedding(1, 384)
    (scaler): PackedStdScaler()
    (in_proj): MultiInSizeLinear(in_features_ls=[8, 16, 32, 64, 128], out_features=384, bias=True, dtype=torch.float32)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): GroupedQueryAttention(
            (var_attn_bias): BinaryAttentionBias(
              (emb): Embedding(2, 6)
            )
            (time_qk_proj): QueryKeyProjection(
              (query_proj): RotaryProjection()
              (key_proj): RotaryProjection()
            )
            (q_proj): Linear(in_features=384, out_features=384, bias=False)
            (k_proj): Linear(in_features=384, out_features=384, bias=False)
            (v_proj): Linear(in_features=384, out_features=384, bias=False)
            (q_norm): RMSNorm(normalized_shape=(64,), eps=1e-05, weight=True)
            (k_norm): RMSNorm(normalized_sh

In [7]:
# Cell 6 - Optional training/evaluation setup or checkpoint loading
# Default path is export-only: load the saved checkpoint and do not build datasets or retrain.
import time

if not RUN_TRAINING and not RUN_EVALUATION:
    # Fast path: load checkpoint only, no dataset building needed (e.g. for Cell 10 export).
    if not os.path.exists(BEST_CHECKPOINT_PATH):
        raise FileNotFoundError(f'RUN_TRAINING=False but checkpoint is missing: {BEST_CHECKPOINT_PATH}')
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
    if 'feature_mean' not in checkpoint or 'feature_std' not in checkpoint:
        raise KeyError('Checkpoint is missing feature_mean/feature_std; cannot export consistently.')
    FEATURE_MEAN = checkpoint['feature_mean']
    FEATURE_STD = checkpoint['feature_std']
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f'Loaded existing checkpoint, skipping dataset build/training/evaluation: {BEST_CHECKPOINT_PATH}')

elif not RUN_TRAINING:
    # Evaluation-only path: load checkpoint for stats, build only test_dataset.
    # Skips train/val dataset building entirely — the main source of the multi-hour wait.
    if not os.path.exists(BEST_CHECKPOINT_PATH):
        raise FileNotFoundError(f'RUN_TRAINING=False but checkpoint is missing: {BEST_CHECKPOINT_PATH}')
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
    if 'feature_mean' not in checkpoint or 'feature_std' not in checkpoint:
        raise KeyError('Checkpoint is missing feature_mean/feature_std; cannot evaluate consistently.')
    FEATURE_MEAN = checkpoint['feature_mean']
    FEATURE_STD = checkpoint['feature_std']
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f'Loaded existing checkpoint: {BEST_CHECKPOINT_PATH}')
    print('Building test dataset only (skipping train/val to save time)...')
    build_start = time.time()
    test_dataset = KalshiMoiraiDataset(test_df, FEATURE_COLUMNS, TARGET_COLUMNS, SEQ_LEN, STRIDE, FEATURE_MEAN, FEATURE_STD, MAX_SEQUENCES, SEED + 2, name='test')
    print(f'Test dataset build time: {(time.time() - build_start) / 60:.2f} min')
    print(f'Test sequences: {len(test_dataset):,}')
    if len(test_dataset) == 0:
        raise ValueError('Test dataset has zero valid sequences.')

else:
    # Full path: compute normalization stats from train data, build all datasets, and train.
    train_feature_frame = train_df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    FEATURE_MEAN = train_feature_frame.mean(axis=0).astype(np.float32).to_numpy()
    FEATURE_STD = train_feature_frame.std(axis=0).replace(0, 1).fillna(1).astype(np.float32).to_numpy()
    del train_feature_frame
    gc.collect()

    print('Building vectorized datasets...')
    build_start = time.time()
    train_dataset = KalshiMoiraiDataset(train_df, FEATURE_COLUMNS, TARGET_COLUMNS, SEQ_LEN, STRIDE, FEATURE_MEAN, FEATURE_STD, MAX_SEQUENCES, SEED, name='train')
    val_dataset = KalshiMoiraiDataset(val_df, FEATURE_COLUMNS, TARGET_COLUMNS, SEQ_LEN, STRIDE, FEATURE_MEAN, FEATURE_STD, MAX_SEQUENCES, SEED + 1, name='val')
    test_dataset = KalshiMoiraiDataset(test_df, FEATURE_COLUMNS, TARGET_COLUMNS, SEQ_LEN, STRIDE, FEATURE_MEAN, FEATURE_STD, MAX_SEQUENCES, SEED + 2, name='test')
    print(f'Dataset build time: {(time.time() - build_start) / 60:.2f} min')
    print(f'Train sequences: {len(train_dataset):,}')
    print(f'Val sequences: {len(val_dataset):,}')
    print(f'Test sequences: {len(test_dataset):,}')
    if min(len(train_dataset), len(val_dataset), len(test_dataset)) == 0:
        raise ValueError('At least one split has zero valid sequences.')

    train_steps_per_epoch = math.ceil(len(train_dataset) / BATCH_SIZE)
    val_steps = math.ceil(len(val_dataset) / BATCH_SIZE)
    print(f'Train steps per epoch: {train_steps_per_epoch:,}')
    print(f'Val steps: {val_steps:,}')
    print(f'Device before training: {device}')
    if torch.cuda.is_available():
        print(f'GPU memory allocated before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

    train_counts = train_dataset.label_counts()
    print('Training label counts by horizon, columns=[down, none, up]:')
    for horizon, counts in zip(HORIZON_NAMES, train_counts):
        print(horizon, counts.tolist())

    class_weights_np = train_counts.sum(axis=1, keepdims=True) / (3.0 * np.maximum(train_counts, 1))
    class_weights_np = class_weights_np / class_weights_np.mean(axis=1, keepdims=True)
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
    criteria = [nn.CrossEntropyLoss(weight=class_weights[h] if USE_CLASS_WEIGHTS else None, label_smoothing=0.02) for h in range(4)]
    print('Class weights:')
    print(class_weights.detach().cpu().numpy())

    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('head.')]
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('head.')]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': head_params, 'lr': LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = max(1, EPOCHS * train_steps_per_epoch)
    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return float(step + 1) / float(max(1, WARMUP_STEPS))
        progress = float(step - WARMUP_STEPS) / float(max(1, total_steps - WARMUP_STEPS))
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda' and USE_AMP))

    def compute_loss(logits, labels):
        return torch.stack([criteria[h](logits[:, h, :], labels[:, h]) for h in range(4)]).mean()

    @torch.no_grad()
    def validate(model, dataset, max_batches=None):
        model.eval()
        all_preds, all_labels = [], []
        total_batches = math.ceil(len(dataset) / BATCH_SIZE)
        for step, (x, y) in iter_batches(dataset, BATCH_SIZE, device, shuffle=False):
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and USE_AMP)):
                logits = model(x)
            preds = logits.argmax(dim=-1)
            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(y.detach().cpu().numpy())
            if step == 1 or step % 500 == 0 or step == total_batches:
                print(f'  validation step {step:,}/{total_batches:,}')
            if max_batches is not None and step >= max_batches:
                break
        y_pred_tmp = np.concatenate(all_preds, axis=0)
        y_true_tmp = np.concatenate(all_labels, axis=0)
        acc = np.array([accuracy_score(y_true_tmp[:, h], y_pred_tmp[:, h]) for h in range(4)])
        macro_f1 = np.array([f1_score(y_true_tmp[:, h], y_pred_tmp[:, h], labels=[0, 1, 2], average='macro', zero_division=0) for h in range(4)])
        pred_counts = np.stack([np.bincount(y_pred_tmp[:, h], minlength=3)[:3] for h in range(4)])
        return acc, macro_f1, pred_counts

    best_mean_val_macro_f1 = -1.0
    global_step = 0
    print(f'Checkpoints will be saved to: {BEST_CHECKPOINT_PATH}')
    print('Starting training...')
    print(f'Epochs: {EPOCHS}')
    print(f'Batch size: {BATCH_SIZE}')
    print(f'Steps per epoch: {train_steps_per_epoch:,}')
    progress_every = 100

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        print(f'\nStarting epoch {epoch}/{EPOCHS}')
        model.train()
        running_loss = 0.0
        seen = 0

        for step, (x, y) in iter_batches(train_dataset, BATCH_SIZE, device, shuffle=True, seed=SEED + epoch):
            if step == 1 and torch.cuda.is_available():
                print(f'GPU memory after first batch transfer: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
            if not torch.isfinite(x).all():
                raise ValueError('Non-finite input batch encountered.')

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and USE_AMP)):
                logits = model(x)
                loss = compute_loss(logits, y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite loss at epoch={epoch}, step={global_step}. Stop and lower LR / disable AMP.')

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            batch = y.size(0)
            running_loss += loss.item() * batch
            seen += batch
            global_step += 1

            if step == 1 or step % progress_every == 0 or step == train_steps_per_epoch:
                elapsed_min = (time.time() - epoch_start) / 60
                avg_loss = running_loss / max(1, seen)
                pct = 100 * step / train_steps_per_epoch
                gpu_mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
                print(f'Epoch {epoch}/{EPOCHS} | step {step:,}/{train_steps_per_epoch:,} ({pct:.1f}%) | loss {avg_loss:.5f} | elapsed {elapsed_min:.1f} min | gpu_mem {gpu_mem:.2f} GB')

        train_loss = running_loss / max(1, seen)
        print(f'Epoch {epoch}/{EPOCHS} training done in {(time.time() - epoch_start) / 60:.2f} min. Running validation...')
        val_acc, val_macro_f1, val_pred_counts = validate(model, val_dataset)
        mean_val_macro_f1 = float(val_macro_f1.mean())
        print(f'Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.5f} | val_macro_f1_mean={mean_val_macro_f1:.4f} | val_acc_mean={float(val_acc.mean()):.4f}')

        if mean_val_macro_f1 > best_mean_val_macro_f1:
            best_mean_val_macro_f1 = mean_val_macro_f1
            torch.save({
                'model_state_dict': model.state_dict(),
                'feature_columns': FEATURE_COLUMNS,
                'target_columns': TARGET_COLUMNS,
                'feature_mean': FEATURE_MEAN,
                'feature_std': FEATURE_STD,
                'moirai_model_id': MOIRAI_MODEL_ID,
                'moirai_kind': MOIRAI_KIND,
                'hyperparameters': {
                    'SEQ_LEN': SEQ_LEN, 'STRIDE': STRIDE, 'PATCH_SIZE': PATCH_SIZE,
                    'LR_BACKBONE': LR_BACKBONE, 'LR_HEAD': LR_HEAD,
                    'FREEZE_BACKBONE': FREEZE_BACKBONE, 'USE_CLASS_WEIGHTS': USE_CLASS_WEIGHTS,
                    'BATCH_SIZE': BATCH_SIZE, 'USE_AMP': USE_AMP,
                },
                'best_mean_val_macro_f1': best_mean_val_macro_f1,
            }, BEST_CHECKPOINT_PATH)
            print(f'  saved best checkpoint to {BEST_CHECKPOINT_PATH}')

Loaded existing checkpoint, skipping dataset build/training/evaluation: /content/drive/MyDrive/CS1090B/project/checkpoints/moirai_classifier_best.pt


In [8]:
# Cell 7 - Test evaluation
if not RUN_EVALUATION:
    print('Skipping Cell 7 test evaluation because RUN_EVALUATION=False. Set RUN_EVALUATION=True in Cell 5 to run metrics.')
else:
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
    FEATURE_MEAN = checkpoint.get('feature_mean', FEATURE_MEAN)
    FEATURE_STD = checkpoint.get('feature_std', FEATURE_STD)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    all_preds, all_labels, all_probs = [], [], []
    test_steps = math.ceil(len(test_dataset) / BATCH_SIZE)
    print(f'Running test evaluation over {len(test_dataset):,} sequences ({test_steps:,} batches)...')
    with torch.no_grad():
        for step, (x, y) in iter_batches(test_dataset, BATCH_SIZE, device, shuffle=False):
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and USE_AMP)):
                logits = model(x)
                probs = torch.softmax(logits, dim=-1)
            all_preds.append(logits.argmax(dim=-1).cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            if step == 1 or step % 500 == 0 or step == test_steps:
                print(f'  test step {step:,}/{test_steps:,}')

    y_pred = np.concatenate(all_preds, axis=0)
    y_prob = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_labels, axis=0)
    summary_rows = []
    for h_idx, horizon in enumerate(HORIZON_NAMES):
        acc = accuracy_score(y_true[:, h_idx], y_pred[:, h_idx])
        macro_f1 = f1_score(y_true[:, h_idx], y_pred[:, h_idx], labels=[0, 1, 2], average='macro', zero_division=0)
        weighted_f1 = f1_score(y_true[:, h_idx], y_pred[:, h_idx], labels=[0, 1, 2], average='weighted', zero_division=0)
        summary_rows.append({'horizon': horizon, 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1})
        print(f'\n=== Horizon {horizon} ===')
        print(f'Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}')
        print(classification_report(y_true[:, h_idx], y_pred[:, h_idx], labels=[0, 1, 2], target_names=CLASS_NAMES, digits=4, zero_division=0))
        print('Confusion matrix rows=true, cols=pred; labels=[down, none, up]')
        print(confusion_matrix(y_true[:, h_idx], y_pred[:, h_idx], labels=[0, 1, 2]))

    summary = pd.DataFrame(summary_rows).set_index('horizon')
    print('\nSummary table:')
    display(summary.T)


Skipping Cell 7 test evaluation because RUN_EVALUATION=False. Set RUN_EVALUATION=True in Cell 5 to run metrics.


In [9]:
# Cell 8 - Inference helper
def predict(trades_df, ticker, model, device, feature_cols=FEATURE_COLUMNS, seq_len=SEQ_LEN, feature_mean=FEATURE_MEAN, feature_std=FEATURE_STD):
    sub = trades_df.loc[trades_df['ticker'] == ticker].sort_values('created_time', kind='mergesort')
    if len(sub) < seq_len:
        raise ValueError(f'Need at least {seq_len} trades for {ticker}; got {len(sub)}.')
    missing = [c for c in feature_cols if c not in sub.columns]
    if missing:
        raise ValueError(f'Missing feature columns: {missing}')
    x = sub.tail(seq_len)[feature_cols].to_numpy(dtype=np.float32, copy=True)
    x = (x - np.asarray(feature_mean, dtype=np.float32)) / np.asarray(feature_std, dtype=np.float32)
    if not np.isfinite(x).all():
        raise ValueError('Recent window contains non-finite normalized features.')
    x = torch.from_numpy(x).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and USE_AMP)):
            probs = torch.softmax(model(x), dim=-1).squeeze(0).cpu().numpy()
    out = {}
    for h_idx, horizon in enumerate(HORIZON_NAMES):
        pred_idx = int(probs[h_idx].argmax())
        out[horizon] = {
            'predicted_class': CLASS_NAMES[pred_idx],
            'prob_down': float(probs[h_idx, 0]),
            'prob_none': float(probs[h_idx, 1]),
            'prob_up': float(probs[h_idx, 2]),
        }
        print(f"{horizon}: {CLASS_NAMES[pred_idx]} | down={probs[h_idx, 0]:.4f}, none={probs[h_idx, 1]:.4f}, up={probs[h_idx, 2]:.4f}")
    return out

# Example:
# sample_ticker = test_df['ticker'].iloc[0]
# preds = predict(test_df, sample_ticker, model, device)

In [10]:
# Cell 9 - Publication figures from existing Moirai evaluation arrays
if not RUN_EVALUATION:
    print('Skipping Cell 9 figures because RUN_EVALUATION=False. Set RUN_EVALUATION=True in Cell 5 and run Cell 7 first.')
else:
    # Run this after Cell 7 has produced y_true, y_pred, and y_prob. It does not rerun inference.
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

    FIGURE_DIR = globals().get('FIGURE_DIR', '/content/drive/MyDrive/CS1090B/project/figures/moirai')
    os.makedirs(FIGURE_DIR, exist_ok=True)

    required = ['y_true', 'y_pred', 'HORIZON_NAMES', 'CLASS_NAMES']
    missing = [name for name in required if name not in globals()]
    if missing:
        raise NameError(f'Missing {missing}. Run Cell 7 first so y_true and y_pred exist.')

    plt.style.use('default')
    COLORS = {'down': '#b91c1c', 'none': '#334155', 'up': '#15803d'}
    CLASS_COLORS = [COLORS['down'], COLORS['none'], COLORS['up']]

    def savefig(fig, name):
        png_path = os.path.join(FIGURE_DIR, f'{name}.png')
        pdf_path = os.path.join(FIGURE_DIR, f'{name}.pdf')
        fig.savefig(png_path, dpi=220, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')
        print(f'Saved {png_path}')
        print(f'Saved {pdf_path}')

    summary_rows = []
    class_metric_rows = []
    conf_mats = {}
    conf_mats_norm = {}
    distribution_rows = []

    for h_idx, horizon in enumerate(HORIZON_NAMES):
        report = classification_report(
            y_true[:, h_idx], y_pred[:, h_idx], labels=[0, 1, 2],
            target_names=CLASS_NAMES, output_dict=True, zero_division=0,
        )
        acc = accuracy_score(y_true[:, h_idx], y_pred[:, h_idx])
        macro_f1 = report['macro avg']['f1-score']
        weighted_f1 = report['weighted avg']['f1-score']
        summary_rows.append({'horizon': horizon, 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1})

        for cls in CLASS_NAMES:
            class_metric_rows.append({
                'horizon': horizon, 'class': cls,
                'precision': report[cls]['precision'],
                'recall': report[cls]['recall'],
                'f1': report[cls]['f1-score'],
                'support': report[cls]['support'],
            })

        cm = confusion_matrix(y_true[:, h_idx], y_pred[:, h_idx], labels=[0, 1, 2])
        conf_mats[horizon] = cm
        conf_mats_norm[horizon] = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

        true_counts = np.bincount(y_true[:, h_idx], minlength=3)[:3]
        pred_counts = np.bincount(y_pred[:, h_idx], minlength=3)[:3]
        for cls_idx, cls in enumerate(CLASS_NAMES):
            distribution_rows.append({'horizon': horizon, 'class': cls, 'kind': 'True', 'share': true_counts[cls_idx] / true_counts.sum()})
            distribution_rows.append({'horizon': horizon, 'class': cls, 'kind': 'Predicted', 'share': pred_counts[cls_idx] / pred_counts.sum()})

    summary = pd.DataFrame(summary_rows).set_index('horizon')
    class_metrics = pd.DataFrame(class_metric_rows)
    distributions = pd.DataFrame(distribution_rows)
    display(summary.T)
    summary.to_csv(os.path.join(FIGURE_DIR, 'moirai_summary_metrics.csv'))
    class_metrics.to_csv(os.path.join(FIGURE_DIR, 'moirai_class_metrics.csv'), index=False)

    # 1. Normalized confusion matrices.
    fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)
    for ax, horizon in zip(axes.ravel(), HORIZON_NAMES):
        data = conf_mats_norm[horizon]
        im = ax.imshow(data, cmap='Blues', vmin=0, vmax=1)
        ax.set_title(f'{horizon} normalized confusion matrix', fontsize=12, weight='bold')
        ax.set_xlabel('Predicted class')
        ax.set_ylabel('True class')
        ax.set_xticks(range(3), CLASS_NAMES)
        ax.set_yticks(range(3), CLASS_NAMES)
        raw = conf_mats[horizon]
        for i in range(3):
            for j in range(3):
                color = 'white' if data[i, j] > 0.55 else '#0f172a'
                ax.text(j, i, f'{data[i, j]:.1%}\n({raw[i, j]:,})', ha='center', va='center', color=color, fontsize=9)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label='Row-normalized share')
    savefig(fig, 'moirai_confusion_matrices_normalized')
    plt.show()

    # 2. Headline metrics.
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(HORIZON_NAMES))
    width = 0.25
    for offset, metric, color in [(-width, 'accuracy', '#2563eb'), (0, 'macro_f1', '#7c3aed'), (width, 'weighted_f1', '#059669')]:
        vals = summary.loc[HORIZON_NAMES, metric].values
        bars = ax.bar(x + offset, vals, width, label=metric.replace('_', ' ').title(), color=color, alpha=0.9)
        ax.bar_label(bars, labels=[f'{v:.3f}' for v in vals], padding=3, fontsize=8)
    ax.set_xticks(x, HORIZON_NAMES)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.set_xlabel('Prediction horizon')
    ax.set_title('Moirai Test Performance by Horizon', fontsize=13, weight='bold')
    ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
    ax.grid(axis='y', alpha=0.25)
    savefig(fig, 'moirai_summary_metrics')
    plt.show()

    # 3. Per-class F1.
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(HORIZON_NAMES))
    width = 0.24
    for idx, cls in enumerate(CLASS_NAMES):
        vals = class_metrics[class_metrics['class'] == cls].set_index('horizon').loc[HORIZON_NAMES, 'f1'].values
        ax.bar(x + (idx - 1) * width, vals, width, label=cls, color=CLASS_COLORS[idx], alpha=0.9)
    ax.set_xticks(x, HORIZON_NAMES)
    ax.set_ylim(0, 1)
    ax.set_ylabel('F1 score')
    ax.set_xlabel('Prediction horizon')
    ax.set_title('Moirai Per-Class F1 Scores', fontsize=13, weight='bold')
    ax.legend(title='Class', frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
    ax.grid(axis='y', alpha=0.25)
    savefig(fig, 'moirai_per_class_f1')
    plt.show()

    # 4. True vs predicted class shares.
    fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True, constrained_layout=True)
    for ax, horizon in zip(axes, HORIZON_NAMES):
        sub = distributions[distributions['horizon'] == horizon]
        for k_idx, kind in enumerate(['True', 'Predicted']):
            vals = sub[sub['kind'] == kind].set_index('class').loc[CLASS_NAMES, 'share'].values
            ax.bar(np.arange(3) + (k_idx - 0.5) * 0.36, vals, width=0.34, label=kind, color=['#94a3b8', '#0f172a'][k_idx], alpha=0.9)
        ax.set_title(horizon, weight='bold')
        ax.set_xticks(range(3), CLASS_NAMES, rotation=30)
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.25)
    axes[0].set_ylabel('Share of examples')
    axes[0].legend(frameon=False, loc='upper left')
    fig.suptitle('Moirai True vs Predicted Class Distribution', fontsize=14, weight='bold')
    savefig(fig, 'moirai_class_distribution')
    plt.show()

    # 5. Mean predicted probability by true class, if probabilities are available.
    if 'y_prob' in globals():
        fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True, sharey=True)
        for ax, h_idx, horizon in zip(axes.ravel(), range(4), HORIZON_NAMES):
            means = []
            for true_cls in range(3):
                mask = y_true[:, h_idx] == true_cls
                means.append(y_prob[mask, h_idx, :].mean(axis=0) if mask.any() else np.zeros(3))
            means = np.vstack(means)
            im = ax.imshow(means, cmap='Greens', vmin=0, vmax=1)
            ax.set_title(f'{horizon}: mean predicted probabilities', fontsize=11, weight='bold')
            ax.set_xlabel('Predicted probability for class')
            ax.set_ylabel('True class')
            ax.set_xticks(range(3), CLASS_NAMES)
            ax.set_yticks(range(3), CLASS_NAMES)
            for i in range(3):
                for j in range(3):
                    ax.text(j, i, f'{means[i, j]:.2f}', ha='center', va='center', color='white' if means[i, j] > 0.5 else '#0f172a')
        fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label='Mean probability')
        savefig(fig, 'moirai_mean_probabilities_by_true_class')
        plt.show()
    else:
        print('Skipping probability heatmap because y_prob is not defined in the current notebook state.')


Skipping Cell 9 figures because RUN_EVALUATION=False. Set RUN_EVALUATION=True in Cell 5 and run Cell 7 first.


In [11]:
# Cell 10 - Export all-trades raw logits to parquet
# Produces one row per valid same-ticker prediction window ending at created_time.
# Column names use the requested probability_* schema, but values are raw logits.
# Fast path: sort once, build valid same-ticker windows globally, and run large GPU batches.
import time
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

MODEL_NAME = 'moirai'
HORIZON_MINUTES = [5, 15, 30, 60]
EXPORT_CLASS_ORDER = [('no_jump', 1), ('up', 2), ('down', 0)]  # model class order is [down, none/no_jump, up]
EXPORT_BATCH_SIZE = 4096  # Reduced to avoid OutOfMemoryError. Previously 65536.
EXPORT_STRIDE = 1        # 1 = every valid trade row. Increase to reduce output size.
EXPORT_MAX_ROWS = None   # Set an int for a capped smoke/full run; None exports everything.
PREDICTION_OUTPUT_PATH = '/content/drive/MyDrive/CS1090B/project/predictions/moirai_all_trades_logits.parquet'

checkpoint_path = BEST_CHECKPOINT_PATH
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f'No Moirai checkpoint found: {checkpoint_path}')
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
if 'feature_mean' in checkpoint and 'feature_std' in checkpoint:
    FEATURE_MEAN = checkpoint['feature_mean']
    FEATURE_STD = checkpoint['feature_std']
elif 'FEATURE_MEAN' not in globals() or 'FEATURE_STD' not in globals():
    raise KeyError('Checkpoint is missing feature_mean/feature_std and no FEATURE_MEAN/FEATURE_STD globals exist.')
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()
print(f'Loaded checkpoint for export: {checkpoint_path}')

expected_output_columns = ['ticker', 'created_time'] + [
    f'probability_{class_name}_{horizon_min}minutes_{MODEL_NAME}'
    for horizon_min in HORIZON_MINUTES
    for class_name, _ in EXPORT_CLASS_ORDER
]


def _table_from_logits(ticker_values, created_values, end_positions, logits, model_name):
    data = {
        'ticker': pa.array(ticker_values[end_positions]),
        'created_time': pa.array(created_values[end_positions]),
    }
    for h_idx, horizon_min in enumerate(HORIZON_MINUTES):
        for class_name, class_idx in EXPORT_CLASS_ORDER:
            data[f'probability_{class_name}_{horizon_min}minutes_{model_name}'] = pa.array(logits[:, h_idx, class_idx])
    return pa.table(data)


def _prepare_fast_export_arrays(trades_df, feature_cols, seq_len, feature_mean, feature_std, export_stride):
    t0 = time.time()
    feature_mean = np.asarray(feature_mean, dtype=np.float32)
    feature_std = np.asarray(feature_std, dtype=np.float32)
    if len(feature_mean) != len(feature_cols) or len(feature_std) != len(feature_cols):
        raise ValueError(
            f'Feature stat length mismatch: mean={len(feature_mean)}, std={len(feature_std)}, features={len(feature_cols)}'
        )
    if np.any(~np.isfinite(feature_mean)) or np.any(~np.isfinite(feature_std)):
        raise ValueError('Feature normalization stats contain NaN or inf.')
    if np.any(feature_std == 0):
        raise ValueError('Feature normalization std contains zero values.')
    if export_stride < 1:
        raise ValueError('EXPORT_STRIDE must be >= 1.')

    print('Sorting trades by ticker/created_time...')
    frame = trades_df.sort_values(['ticker', 'created_time'], kind='mergesort').reset_index(drop=True)
    n_rows = len(frame)
    print(f'Sorted {n_rows:,} rows in {(time.time() - t0) / 60:.2f} min')

    print('Building normalized feature matrix...')
    features = frame[feature_cols].to_numpy(dtype=np.float32, copy=True)
    features -= feature_mean
    features /= feature_std
    valid_rows = np.isfinite(features).all(axis=1)
    valid_prefix = np.concatenate([[0], np.cumsum(valid_rows, dtype=np.int32)])
    print(f'Feature matrix shape: {features.shape}; finite rows: {valid_rows.sum():,}')

    print('Vectorizing valid same-ticker window endpoints...')
    ticker_codes = pd.factorize(frame['ticker'], sort=False)[0]
    change_points = np.where(np.diff(ticker_codes) != 0)[0] + 1
    group_starts = np.concatenate([[0], change_points])
    group_ends = np.concatenate([change_points, [n_rows]])

    end_chunks = []
    for s, e in zip(group_starts, group_ends):
        if e - s < seq_len:
            continue
        end_pos = np.arange(s + seq_len - 1, e, export_stride, dtype=np.int64)
        start_pos = end_pos - seq_len + 1
        valid_windows = (valid_prefix[end_pos + 1] - valid_prefix[start_pos]) == seq_len
        if valid_windows.any():
            end_chunks.append(end_pos[valid_windows])

    if not end_chunks:
        raise ValueError('No valid prediction windows were found in all_trades_features.')
    end_positions = np.concatenate(end_chunks)
    if EXPORT_MAX_ROWS is not None:
        end_positions = end_positions[:int(EXPORT_MAX_ROWS)]
    print(f'Prepared {len(end_positions):,} prediction windows in {(time.time() - t0) / 60:.2f} min')

    ticker_values = frame['ticker'].astype(str).to_numpy(copy=False)
    created_values = frame['created_time'].to_numpy(copy=False)
    return features, ticker_values, created_values, end_positions


def export_all_trade_logits_to_parquet_fast(
    trades_df,
    model,
    output_path,
    model_name,
    feature_cols=FEATURE_COLUMNS,
    seq_len=SEQ_LEN,
    feature_mean=FEATURE_MEAN,
    feature_std=FEATURE_STD,
    batch_size=EXPORT_BATCH_SIZE,
    export_stride=EXPORT_STRIDE,
    report_every=1_000_000,
):
    if not output_path.startswith('/content/drive/MyDrive/'):
        raise ValueError(f'Output path should be on Google Drive, got: {output_path}')
    missing = [c for c in ['ticker', 'created_time', *feature_cols] if c not in trades_df.columns]
    if missing:
        raise ValueError(f'Missing required export columns: {missing}')

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Write the temp file to local /content/ SSD to avoid thousands of slow Drive FUSE writes.
    # A single shutil.move at the end copies the finished file to Drive.
    local_tmp = '/content/' + os.path.basename(output_path) + '.tmp'
    smoke_path = '/content/' + os.path.basename(output_path) + '.smoke_test.parquet'
    for stale_path in [local_tmp, smoke_path]:
        if os.path.exists(stale_path):
            os.remove(stale_path)

    free_gb = shutil.disk_usage('/content').free / 1e9
    print(f'Local /content free disk before export: {free_gb:.1f} GB')
    print(f'Export stride={export_stride}, max_rows={EXPORT_MAX_ROWS}, batch_size={batch_size}')

    features, ticker_values, created_values, end_positions = _prepare_fast_export_arrays(
        trades_df=trades_df,
        feature_cols=feature_cols,
        seq_len=seq_len,
        feature_mean=feature_mean,
        feature_std=feature_std,
        export_stride=export_stride,
    )

    window_offsets = np.arange(seq_len, dtype=np.int64)
    writer = None
    total_rows = 0
    next_report = report_every
    t0 = time.time()

    model.eval()
    try:
        with torch.no_grad():
            for batch_start in range(0, len(end_positions), batch_size):
                batch_end_positions = end_positions[batch_start:batch_start + batch_size]
                rows = batch_end_positions[:, None] - seq_len + 1 + window_offsets[None, :]
                x = torch.from_numpy(features[rows]).to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and USE_AMP)):
                    logits = model(x).detach().cpu().numpy().astype(np.float32)
                if logits.shape != (len(batch_end_positions), len(HORIZON_MINUTES), len(CLASS_NAMES)):
                    raise ValueError(f'Unexpected logits shape: {logits.shape}')

                table = _table_from_logits(ticker_values, created_values, batch_end_positions, logits, model_name)
                if table.column_names != expected_output_columns:
                    raise ValueError(f'Unexpected output columns: {table.column_names}')

                if writer is None:
                    # Pass the expected column names to the ParquetWriter schema argument.
                    # The table object itself already has the correct schema from _table_from_logits.
                    writer = pq.ParquetWriter(local_tmp, table.schema, compression='snappy')
                    pq.write_table(table.slice(0, min(10, table.num_rows)), smoke_path, compression='snappy')
                    smoke_table = pq.read_table(smoke_path)
                    if smoke_table.column_names != expected_output_columns:
                        raise ValueError(f'Smoke-test parquet columns changed: {smoke_table.column_names}')
                    os.remove(smoke_path)
                    print('Parquet smoke test passed.')

                writer.write_table(table)
                torch.cuda.empty_cache()
                total_rows += table.num_rows
                if total_rows >= next_report:
                    elapsed = (time.time() - t0) / 60
                    rate = total_rows / max(elapsed * 60, 1e-9)
                    print(f'Wrote {total_rows:,} prediction rows in {elapsed:.1f} min ({rate:,.0f} rows/sec)...')
                    next_report += report_every
    finally:
        if writer is not None:
            writer.close()

    if total_rows == 0:
        if os.path.exists(local_tmp):
            os.remove(local_tmp)
        raise ValueError('No rows were written to the prediction parquet.')

    print(f'Inference done: {total_rows:,} rows in {(time.time() - t0) / 60:.2f} min. Copying to Drive...')
    copy_start = time.time()
    if os.path.exists(output_path):
        os.remove(output_path)
    shutil.move(local_tmp, output_path)
    print(f'Copied to Drive in {(time.time() - copy_start) / 60:.2f} min: {output_path}')
    return output_path


# Validate the parquet schema before loading the full table into memory.
parquet_schema = pq.ParquetFile(PARQUET_PATH).schema_arrow
missing_in_file = [c for c in ['ticker', 'created_time', *FEATURE_COLUMNS] if c not in parquet_schema.names]
if missing_in_file:
    raise ValueError(f'Missing columns in PARQUET_PATH: {missing_in_file}')

all_trades_for_export = pd.read_parquet(PARQUET_PATH, columns=['ticker', 'created_time', *FEATURE_COLUMNS])
export_path = export_all_trade_logits_to_parquet_fast(
    all_trades_for_export,
    model=model,
    output_path=PREDICTION_OUTPUT_PATH,
    model_name=MODEL_NAME,
)
del all_trades_for_export
gc.collect()
print(export_path)


Loaded checkpoint for export: /content/drive/MyDrive/CS1090B/project/checkpoints/moirai_classifier_best.pt
Local /content free disk before export: 204.0 GB
Export stride=1, max_rows=None, batch_size=4096
Sorting trades by ticker/created_time...
Sorted 63,801,798 rows in 0.83 min
Building normalized feature matrix...
Feature matrix shape: (63801798, 40); finite rows: 54,391,664
Vectorizing valid same-ticker window endpoints...
Prepared 44,267,126 prediction windows in 1.07 min
Parquet smoke test passed.
Wrote 1,003,520 prediction rows in 3.2 min (5,270 rows/sec)...
Wrote 2,002,944 prediction rows in 6.3 min (5,278 rows/sec)...
Wrote 3,002,368 prediction rows in 9.5 min (5,281 rows/sec)...
Wrote 4,001,792 prediction rows in 12.6 min (5,282 rows/sec)...
Wrote 5,001,216 prediction rows in 15.8 min (5,283 rows/sec)...
Wrote 6,000,640 prediction rows in 18.9 min (5,284 rows/sec)...
Wrote 7,000,064 prediction rows in 22.1 min (5,285 rows/sec)...
Wrote 8,003,584 prediction rows in 25.2 min (5,